## Section 0 — Imports & Configuration
___

This notebook builds the Stage 1 ML forecasting pipeline for twelve macroeconomic
variables (six owned by Andres, six by Ashwin) using a strictly univariate approach.
The methodology follows a three-step structure:

**Step 1** — lag selection and hyperparameter tuning on pre-COVID training data (earliest
available to 2019 Q4).

**Step 2** — single-origin static evaluation on a COVID-free test set (2022 Q1 to 2025 Q4),
selecting the winning model per variable.

**Step 3** — final 20-quarter forecast (2026 Q1 to 2030 Q4) using re-tuned hyperparameters
on the full COVID-excluded dataset.

**COVID treatment**: the 2020 Q1 to 2021 Q4 window (8 quarters) is excluded entirely from
all three steps and the series is re-indexed so 2022 Q1 directly follows 2019 Q4, consistent
with the missing-observation treatment recommended by Lenza & Primiceri (2022). This differs
deliberately from the Stage 2 PD model, which retains COVID via spline adjustment rather
than exclusion — see methodology note in Section 9 for the justification of this distinction.

**Model pool**: seven models across three families — LASSO, Ridge, Elastic Net
(regularised linear); Random Forest, XGBoost (tree ensembles); SVR, Kernel Ridge Regression
(kernel methods). MLP/DNN, LSTM, and KNN are excluded — see methods tally discussion.

**References**: Bank & Eder (2021) SSRN 3981339 · Goulet Coulombe et al. (2022) JAE ·
Lenza & Primiceri (2022) JAE · Makridakis, Spiliotis & Assimakopoulos (2018) IJF

In [2]:
# ─────────────────────────────────────────────────────────────────────────────
# 02b — STAGE 1 ML FORECASTING — US QUARTERLY
# ─────────────────────────────────────────────────────────────────────────────
# Univariate ML forecasting of macroeconomic variables, full individual history
# per variable. COVID window (2020 Q1 - 2021 Q4) excluded and re-indexed per
# Csápai (2023); Lenza & Primiceri (2022).
# ─────────────────────────────────────────────────────────────────────────────

import pandas as pd
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from statsmodels.tsa.stattools import adfuller, kpss, acf, pacf
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import TimeSeriesSplit, GridSearchCV
from sklearn.linear_model import LassoCV, RidgeCV, ElasticNetCV, Lasso, Ridge, ElasticNet
from sklearn.kernel_ridge import KernelRidge
from sklearn.svm import SVR
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error
from scipy import stats

import warnings
warnings.filterwarnings('ignore')

# ── Global configuration ──────────────────────────────────────────────────────
GITHUB_RAW = 'https://raw.githubusercontent.com/hogandan85/ST-498/refs/heads/main/Data%20Collection'
DATA_CUTOFF = pd.Timestamp('2025-12-31')

COVID_START = pd.Timestamp('2020-03-31')   # 2020 Q1
COVID_END   = pd.Timestamp('2021-12-31')   # 2021 Q4

TRAIN_END   = pd.Timestamp('2019-12-31')   # Step 1 training cutoff
TEST_START  = pd.Timestamp('2022-03-31')   # Step 2 test start (post-COVID)
TEST_END    = pd.Timestamp('2025-12-31')   # Step 2 test end
YOY_SUBWINDOW_START = pd.Timestamp('2023-03-31')  # base-effect-free sub-window

FORECAST_HORIZON = 20   # quarters, to 2030 Q4

RANDOM_STATE = 42
CV_SPLITS = 5

# ── Colour palette (consistent with EDA/Time Series notebooks) ──────────────
NAVY, BLUE, AMBER, RED, GREY, TEAL = '#1F3864', '#2E86C1', '#E67E22', '#C0392B', '#7F8C8D', '#17A589'
TEMPLATE = 'plotly_white'

print('Section 0 — imports and configuration loaded.')
print(f'Data cutoff       : {DATA_CUTOFF.date()}')
print(f'COVID exclusion   : {COVID_START.date()} to {COVID_END.date()}')
print(f'Step 1 training   : earliest available -> {TRAIN_END.date()}')
print(f'Step 2 test       : {TEST_START.date()} -> {TEST_END.date()}')
print(f'Forecast horizon  : {FORECAST_HORIZON} quarters (2026 Q1 -> 2030 Q4)')

Section 0 — imports and configuration loaded.
Data cutoff       : 2025-12-31
COVID exclusion   : 2020-03-31 to 2021-12-31
Step 1 training   : earliest available -> 2019-12-31
Step 2 test       : 2022-03-31 -> 2025-12-31
Forecast horizon  : 20 quarters (2026 Q1 -> 2030 Q4)


## Section 1 — Data Loading
___

Data is read directly from `MacroVariables_US_Q.csv` — **not** from 
`EDA_analytical_US_Q.csv`. This is a deliberate choice: the EDA's analytical panel is 
truncated to a common start date (1990) because every one of its analytical steps 
(CCF, stationarity-vs-target, recession stress) is anchored to the target variable's 
availability (`us_delinquency_rate`, available from 1991 Q1).

Stage 1 forecasting has no such constraint — each variable should be forecast using its 
own maximum available history, completely independent of any other variable's 
availability. `MacroVariables_US_Q.csv` preserves this, with series ranging from 
`us_industrial_production` (1919) to `us_reer` (1994).

An explicit cutoff of `pd.Timestamp('2025-12-31')` is applied throughout to prevent any 
Q1 2026 data leakage.

In [3]:
# ─────────────────────────────────────────────────────────────────────────────
# Section 1 — Data Loading (full history, no common-panel truncation)
# ─────────────────────────────────────────────────────────────────────────────

raw = pd.read_csv(f'{GITHUB_RAW}/MacroVariables_US_Q.csv', index_col=0, parse_dates=True)
raw = raw[raw.index <= DATA_CUTOFF]
raw = raw.sort_index()

print(f'Raw dataset loaded: {raw.shape}')
print(f'Index range: {raw.index.min().date()} to {raw.index.max().date()}')

# ── Validate per-variable availability — all 12 variables ───────────────────
ALL_RAW_VARS = [
    # Andres
    'us_credit', 'us_sp500_close', 'us_vix', 'us_bond_yield_10y',
    'us_oil_price', 'us_reer',
    # Ashwin
    'us_real_gdp', 'us_unemployment', 'us_cpi', 'us_consumer_confidence',
    'us_house_price_idx', 'us_industrial_production',
]

print('\nPer-variable availability (all 12 raw series):')
for raw_col in ALL_RAW_VARS:
    s = raw[raw_col].dropna()
    print(f'  {raw_col:25s}  N={len(s):4d}  first={s.index.min().date()}  last={s.index.max().date()}')

Raw dataset loaded: (428, 22)
Index range: 1919-03-31 to 2025-12-31

Per-variable availability (all 12 raw series):
  us_credit                  N= 321  first=1945-12-31  last=2025-12-31
  us_sp500_close             N= 304  first=1950-03-31  last=2025-12-31
  us_vix                     N= 144  first=1990-03-31  last=2025-12-31
  us_bond_yield_10y          N= 291  first=1953-06-30  last=2025-12-31
  us_oil_price               N= 160  first=1986-03-31  last=2025-12-31
  us_reer                    N= 128  first=1994-03-31  last=2025-12-31
  us_real_gdp                N= 316  first=1947-03-31  last=2025-12-31
  us_unemployment            N= 312  first=1948-03-31  last=2025-12-31
  us_cpi                     N= 312  first=1948-03-31  last=2025-12-31
  us_consumer_confidence     N= 284  first=1955-03-31  last=2025-12-31
  us_house_price_idx         N= 224  first=1970-03-31  last=2025-12-31
  us_industrial_production   N= 428  first=1919-03-31  last=2025-12-31


## Section 2 — Variable Configuration & Transformation
___

Each of the twelve variables is forecast in its **stationary transformed form**, not as
a raw level. Raw price/index levels (oil, house prices, GDP, etc.) are I(1) and carry
stochastic trends that produce unreliable ML forecasts. Forecasting derived variables
also preserves definitional consistency for downstream stress testing.

Transformations applied, consistent with `01_EDA.ipynb` Section 2:

| Type | Variables |
|---|---|
| YoY % growth (4-quarter lookback) | GDP, house prices, industrial production, oil |
| Log return (1-quarter lookback) | S&P 500, VIX |
| QoQ % growth (1-quarter lookback) | Credit |
| First difference | REER |
| Levels | Bond yield (trend-stationary exception), unemployment, consumer confidence, CPI |

**Structural breaks identified**: `us_credit_qoq_growth` and `us_gdp_yoy_growth` both
showed a significant mean-shift between the pre-1990 and post-1990 regimes, failing
stationarity tests over the full history but passing cleanly when restricted to 1990
onward. Both are therefore restricted to a 1990 Q1 start date. The transformation is
applied to the full raw history first, then the start-date restriction is applied — this
avoids unnecessarily discarding the lookback quarters needed to compute the first valid
transformed value.

**`us_cpi`** arrives from the source data already expressed as YoY inflation, not as a
raw CPI index level — confirmed by inspecting the raw values directly (small positive and
negative percentages consistent with historical inflation/deflation episodes, not an
index in the hundreds). It is therefore configured with `transform: 'level'` rather than
`'yoy'`; applying a further YoY transformation would compute year-over-year growth of an
already-YoY series, which is not economically meaningful and produces unstable values
when the YoY denominator approaches zero. `us_cpi`'s stationarity classification is
inconclusive rather than clearly stationary — see the note below for the reasoning and
the decision to retain it on its full history regardless.

**`us_bond_yield_10y`** remains the one documented exception forecast in levels despite
failing stationarity tests, per the trend-stationary justification established in the EDA
and required for IFRS 9 PD modelling purposes.

In [4]:
# ─────────────────────────────────────────────────────────────────────────────
# Section 2 — Variable Configuration & Transformation (12 variables)
# ─────────────────────────────────────────────────────────────────────────────

VARIABLES = {
    'us_credit_qoq_growth':    {'raw_col': 'us_credit',                 'transform': 'qoq',     'yoy_caveat': False, 'start_date': pd.Timestamp('1990-01-01')},
    'us_sp500_log_ret':        {'raw_col': 'us_sp500_close',            'transform': 'log_ret', 'yoy_caveat': False, 'start_date': None},
    'us_vix_log_ret':          {'raw_col': 'us_vix',                    'transform': 'log_ret', 'yoy_caveat': False, 'start_date': None},
    'us_bond_yield_10y':       {'raw_col': 'us_bond_yield_10y',         'transform': 'level',   'yoy_caveat': False, 'start_date': None},
    'us_oil_yoy':              {'raw_col': 'us_oil_price',              'transform': 'yoy',     'yoy_caveat': True,  'start_date': None},
    'us_reer_diff':            {'raw_col': 'us_reer',                   'transform': 'diff',    'yoy_caveat': False, 'start_date': None},

    'us_gdp_yoy_growth':       {'raw_col': 'us_real_gdp', 'transform': 'yoy', 'yoy_caveat': True, 'start_date': pd.Timestamp('1990-01-01')},
    'us_cpi': {'raw_col': 'us_cpi', 'transform': 'level', 'yoy_caveat': False, 'start_date': None},
    'us_unemployment':         {'raw_col': 'us_unemployment',           'transform': 'level',   'yoy_caveat': False, 'start_date': None},
    'us_consumer_confidence':  {'raw_col': 'us_consumer_confidence',    'transform': 'level',   'yoy_caveat': False, 'start_date': None},
    'us_house_price_yoy':      {'raw_col': 'us_house_price_idx',        'transform': 'yoy',     'yoy_caveat': True,  'start_date': None},
    'us_indprod_yoy':          {'raw_col': 'us_industrial_production',  'transform': 'yoy',     'yoy_caveat': True,  'start_date': None},
}

def apply_transform(series, transform):
    if transform == 'qoq':
        return series.pct_change(1) * 100
    elif transform == 'log_ret':
        return np.log(series / series.shift(1))
    elif transform == 'yoy':
        return series.pct_change(4) * 100
    elif transform == 'diff':
        interp = series.interpolate(method='linear', limit_direction='both')
        return interp.diff()
    elif transform == 'level':
        return series.copy()
    else:
        raise ValueError(f'Unknown transform: {transform}')

series_dict = {}
transform_summary = []

for var_name, cfg in VARIABLES.items():
    raw_series = raw[cfg['raw_col']].dropna()
    raw_series = raw_series[raw_series.index <= DATA_CUTOFF]

    stat_series = apply_transform(raw_series, cfg['transform']).dropna()

    # Apply start_date filter AFTER transformation, not before
    if cfg['start_date'] is not None:
        stat_series = stat_series[stat_series.index >= cfg['start_date']]

    adf_stat, adf_p, *_ = adfuller(stat_series, autolag='AIC')
    kpss_stat, kpss_p, *_ = kpss(stat_series, regression='c', nlags='auto')

    if adf_p < 0.05 and kpss_p > 0.05:
        decision = 'STATIONARY'
    elif adf_p >= 0.05 and kpss_p <= 0.05:
        decision = 'NON-STATIONARY'
    else:
        decision = 'INCONCLUSIVE'

    series_dict[var_name] = stat_series

    transform_summary.append({
        'Variable':   var_name,
        'Raw series': cfg['raw_col'],
        'Transform':  cfg['transform'],
        'N (full)':   len(stat_series),
        'First obs':  stat_series.index.min().date(),
        'Last obs':   stat_series.index.max().date(),
        'ADF p':      round(adf_p, 4),
        'KPSS p':     round(kpss_p, 4),
        'Decision':   decision,
    })

summary_df = pd.DataFrame(transform_summary)

def style_decision(v):
    if v == 'STATIONARY':     return 'background-color:#D5F5E3; color:#1a1a1a; font-weight:bold'
    if v == 'NON-STATIONARY': return 'background-color:#FADBD8; color:#1a1a1a; font-weight:bold'
    return 'background-color:#FEF9E7; color:#1a1a1a'

display(summary_df.style
    .map(style_decision, subset=['Decision'])
    .set_caption('Table 2.1 — Variable Preparation & Stationarity: All Twelve Variables')
    .hide(axis='index'))

Variable,Raw series,Transform,N (full),First obs,Last obs,ADF p,KPSS p,Decision
us_credit_qoq_growth,us_credit,qoq,144,1990-03-31,2025-12-31,0.022400,0.100000,STATIONARY
us_sp500_log_ret,us_sp500_close,log_ret,303,1950-06-30,2025-12-31,0.000000,0.100000,STATIONARY
us_vix_log_ret,us_vix,log_ret,143,1990-06-30,2025-12-31,0.000000,0.100000,STATIONARY
us_bond_yield_10y,us_bond_yield_10y,level,291,1953-06-30,2025-12-31,0.458100,0.010000,NON-STATIONARY
us_oil_yoy,us_oil_price,yoy,156,1987-03-31,2025-12-31,0.004700,0.100000,STATIONARY
us_reer_diff,us_reer,diff,127,1994-06-30,2025-12-31,0.000000,0.100000,STATIONARY
us_gdp_yoy_growth,us_real_gdp,yoy,144,1990-03-31,2025-12-31,0.048100,0.100000,STATIONARY
us_cpi,us_cpi,level,312,1948-03-31,2025-12-31,0.418100,0.100000,INCONCLUSIVE
us_unemployment,us_unemployment,level,312,1948-03-31,2025-12-31,0.001500,0.100000,STATIONARY
us_consumer_confidence,us_consumer_confidence,level,284,1955-03-31,2025-12-31,0.000000,0.100000,STATIONARY


### Note — `us_cpi` Stationarity: Inconclusive Result

Eleven of the twelve variables resolve cleanly to a stationary or non-stationary
classification under the joint ADF/KPSS test. `us_cpi`, however, returns an
inconclusive result (ADF p = 0.4181, KPSS p = 0.10) even after testing truncated
start dates from 1985 through 2000 — none resolve the ambiguity, and ADF's
p-value does not improve monotonically as older data is excluded, which rules
out a simple structural break of the kind affecting `us_credit_qoq_growth` and
`us_gdp_yoy_growth`.

This pattern is consistent with the well-documented persistence of US
inflation: Stock and Watson (2011) report the same ambiguity when applying an
ADF test to US inflation, noting that while a unit root is not formally
rejected at conventional levels, inflation's long-run swings are better
characterised as highly persistent rather than truly non-stationary. Given
this, `us_cpi` is retained on its full available history in levels (it arrives
from the source data already expressed as YoY inflation), in the same spirit
as the trend-stationary treatment of `us_bond_yield_10y` — both variables fail
a strict stationarity classification, but for different and independently
justified reasons.

## Section 3 — COVID Exclusion & Re-indexing
___

The COVID window (2020 Q1 to 2021 Q4, 8 quarters) is removed entirely from each
variable's series prior to any modelling. The re-indexed series treats 2022 Q1 as the
direct successor of 2019 Q4, so no COVID-era values appear as targets or features at
any stage of Steps 1, 2, or 3. This is consistent with the missing-observation treatment
recommended by Lenza & Primiceri (2022) as best practice for macro models estimated
across the COVID period.

**Train/test split** (identical structure for all twelve variables):
- **Training**: earliest available history → 2019 Q4 (COVID-free by construction)
- **Test**: 2022 Q1 → 2025 Q4, 16 quarters (COVID-free by construction)

**Residual YoY base-effect limitation**: for variables using a YoY transformation, the
2022 Q1–Q4 target values are calculated relative to the same quarter in 2021 — placing
excluded COVID-era levels in the YoY denominator. This is an unavoidable consequence of
the YoY definition. Affected variables (`us_oil_yoy`, `us_gdp_yoy_growth`,
`us_house_price_yoy`, `us_indprod_yoy`) use a 12-quarter primary evaluation
sub-window (2023 Q1–2025 Q4) for model selection in Step 2, addressed in Section 5.

In [5]:
# ─────────────────────────────────────────────────────────────────────────────
# Section 3 — COVID Exclusion & Re-indexing (all twelve variables)
# ─────────────────────────────────────────────────────────────────────────────

excl_dict  = {}
train_dict = {}
test_dict  = {}
split_summary = []

for var_name, cfg in VARIABLES.items():
    series = series_dict[var_name]

    covid_mask = (series.index >= COVID_START) & (series.index <= COVID_END)
    excl       = series[~covid_mask].copy()

    train = excl[excl.index <= TRAIN_END]
    test  = excl[(excl.index >= TEST_START) & (excl.index <= TEST_END)]

    excl_dict[var_name]  = excl
    train_dict[var_name] = train
    test_dict[var_name]  = test

    split_summary.append({
        'Variable':          var_name,
        'COVID removed (Q)': int(covid_mask.sum()),
        'YoY caveat':        'Yes' if cfg['yoy_caveat'] else 'No',
        'N train':           len(train),
        'Train start':       train.index.min().date(),
        'Train end':         train.index.max().date(),
        'N test':            len(test),
        'Test start':        test.index.min().date(),
        'Test end':          test.index.max().date(),
    })

split_df = pd.DataFrame(split_summary)

def style_caveat(v):
    if v == 'Yes': return 'background-color:#FEF9E7; color:#1a1a1a'
    return ''

display(split_df.style
    .map(style_caveat, subset=['YoY caveat'])
    .set_caption('Table 3.1 — COVID Exclusion & Train/Test Split: All Twelve Variables')
    .hide(axis='index'))

Variable,COVID removed (Q),YoY caveat,N train,Train start,Train end,N test,Test start,Test end
us_credit_qoq_growth,8,No,120,1990-03-31,2019-12-31,16,2022-03-31,2025-12-31
us_sp500_log_ret,8,No,279,1950-06-30,2019-12-31,16,2022-03-31,2025-12-31
us_vix_log_ret,8,No,119,1990-06-30,2019-12-31,16,2022-03-31,2025-12-31
us_bond_yield_10y,8,No,267,1953-06-30,2019-12-31,16,2022-03-31,2025-12-31
us_oil_yoy,8,Yes,132,1987-03-31,2019-12-31,16,2022-03-31,2025-12-31
us_reer_diff,8,No,103,1994-06-30,2019-12-31,16,2022-03-31,2025-12-31
us_gdp_yoy_growth,8,Yes,120,1990-03-31,2019-12-31,16,2022-03-31,2025-12-31
us_cpi,8,No,288,1948-03-31,2019-12-31,16,2022-03-31,2025-12-31
us_unemployment,8,No,288,1948-03-31,2019-12-31,16,2022-03-31,2025-12-31
us_consumer_confidence,8,No,260,1955-03-31,2019-12-31,16,2022-03-31,2025-12-31


## Section 4 — Step 1: Lag Selection, Training & Hyperparameter Tuning
___

**Lag selection**: for each variable, PACF is computed on the Step 1 training data 
to determine a variable-specific maximum lag ceiling. The 95% confidence interval 
threshold (±1.96/√N) identifies which lags carry significant partial autocorrelation 
once shorter lags are already accounted for. All lags up to the highest significant 
lag are included as candidate features — the effective lag length used by each model 
is an output of training (via regularisation shrinkage or tree-split selection), not 
a fixed input.

**Feature construction**: each variable is converted into a standard supervised 
learning problem — the target is the value at time *t*, and the features are its own 
lagged values (*t-1* through the lag ceiling). This is the only information available 
to each model under the strictly univariate approach; no cross-series features are used.

**Cross-validation**: `TimeSeriesSplit` with k=5 folds is used for all hyperparameter 
selection, preserving temporal ordering with no look-ahead bias.

**Models tuned** (seven, three families):
- Regularised linear — LASSO (`LassoCV`), Ridge (`RidgeCV`), Elastic Net 
  (`ElasticNetCV`) — cross-validate their own penalty automatically
- Kernel methods — Kernel Ridge Regression (RBF/polynomial kernels), SVR (RBF 
  kernel) — grid search over regularisation and kernel parameters
- Tree ensembles — Random Forest, XGBoost — grid search over depth, estimators, 
  and learning rate (XGBoost only)

All features are standardised to zero mean and unit variance before fitting every 
model — required for the penalised linear methods and applied consistently across 
the full pool for simplicity.

**Output**: one set of frozen hyperparameters per model per variable, carried forward 
unchanged to Step 2 (Section 5) for out-of-sample evaluation.

**References**: Goulet Coulombe et al. (2022) JAE — univariate ML benchmark design;
TimeSeriesSplit cross-validation and feature standardisation follow standard practice
for time series ML (Bergmeir, Hyndman & Koo, 2018, *Computational Statistics & Data
Analysis* — validity of cross-validation for autoregressive time series prediction).

In [8]:
# ─────────────────────────────────────────────────────────────────────────────
# Section 4 — Step 1: Lag Selection, Feature Matrix & Hyperparameter Tuning
# ─────────────────────────────────────────────────────────────────────────────

def make_features(series, n_lags):
    """Build supervised learning feature matrix from univariate time series."""
    df = pd.DataFrame({'y': series})
    for lag in range(1, n_lags + 1):
        df[f'lag_{lag}'] = series.shift(lag)
    return df.dropna()

scaler_dict   = {}
lag_dict      = {}
frozen_models = {}
lag_summary   = []

for var_name in VARIABLES:
    print(f'\n{"─"*60}')
    print(f'  {var_name}')
    print(f'{"─"*60}')

    train = train_dict[var_name]
    tscv  = TimeSeriesSplit(n_splits=CV_SPLITS)
    CI_95 = 1.96 / np.sqrt(len(train))

    # ── Lag selection via PACF ────────────────────────────────────────────────
    max_lag   = 8   # 2 years of quarterly history — fixed practical cap
    pacf_vals = pacf(train, nlags=max_lag, alpha=None)
    sig_lags  = [lag for lag in range(1, len(pacf_vals))
                 if abs(pacf_vals[lag]) > CI_95]
    lag_ceil  = max(sig_lags) if sig_lags else 2
    lag_dict[var_name] = lag_ceil
    print(f'  Significant PACF lags: {sig_lags}  →  ceiling: {lag_ceil}')

    # ── Feature matrix ────────────────────────────────────────────────────────
    train_df = make_features(train, lag_ceil)
    X_train  = train_df.drop(columns='y').values
    y_train  = train_df['y'].values

    test       = test_dict[var_name]
    combined   = pd.concat([train.iloc[-lag_ceil:], test])
    test_df    = make_features(combined, lag_ceil)
    X_test     = test_df.drop(columns='y').values
    y_test     = test_df['y'].values

    # ── Standardise ───────────────────────────────────────────────────────────
    scaler     = StandardScaler()
    X_train_sc = scaler.fit_transform(X_train)
    X_test_sc  = scaler.transform(X_test)
    scaler_dict[var_name] = scaler

    # ── Fit all seven models ──────────────────────────────────────────────────
    models = {}

    m = LassoCV(cv=tscv, max_iter=10000, random_state=RANDOM_STATE)
    m.fit(X_train_sc, y_train)
    models['LASSO'] = m
    print(f'  LASSO        alpha={m.alpha_:.4f}')

    m = RidgeCV(alphas=np.logspace(-2, 4, 24), cv=tscv)
    m.fit(X_train_sc, y_train)
    models['Ridge'] = m
    print(f'  Ridge        alpha={m.alpha_:.4f}')

    m = ElasticNetCV(cv=tscv, max_iter=10000, random_state=RANDOM_STATE)
    m.fit(X_train_sc, y_train)
    models['Elastic Net'] = m
    print(f'  Elastic Net  alpha={m.alpha_:.4f}  l1_ratio={m.l1_ratio_:.4f}')

    g = GridSearchCV(KernelRidge(),
        {'alpha': [0.001, 0.01, 0.1, 1, 10],
         'kernel': ['rbf', 'polynomial'],
         'gamma': [0.0001, 0.001, 0.01, 0.1, 1]},
        cv=tscv, scoring='neg_mean_squared_error')
    g.fit(X_train_sc, y_train)
    models['KRR'] = g.best_estimator_
    print(f'  KRR          {g.best_params_}')

    g = GridSearchCV(SVR(kernel='rbf'),
        {'C': [0.1, 1, 10, 100, 1000, 10000], 'epsilon': [0.01, 0.1, 1]},
        cv=tscv, scoring='neg_mean_squared_error')
    g.fit(X_train_sc, y_train)
    models['SVR'] = g.best_estimator_
    print(f'  SVR          {g.best_params_}')

    g = GridSearchCV(RandomForestRegressor(random_state=RANDOM_STATE),
        {'n_estimators': [100, 200],
         'max_depth': [3, 5, None],
         'min_samples_leaf': [2, 5]},
        cv=tscv, scoring='neg_mean_squared_error')
    g.fit(X_train_sc, y_train)
    models['Random Forest'] = g.best_estimator_
    print(f'  Random Forest {g.best_params_}')

    g = GridSearchCV(
        XGBRegressor(random_state=RANDOM_STATE, verbosity=0),
        {'max_depth': [3, 5],
         'learning_rate': [0.05, 0.1],
         'n_estimators': [100, 200],
         'subsample': [0.8, 1.0]},
        cv=tscv, scoring='neg_mean_squared_error')
    g.fit(X_train_sc, y_train)
    models['XGBoost'] = g.best_estimator_
    print(f'  XGBoost      {g.best_params_}')

    frozen_models[var_name] = models

    lag_summary.append({
        'Variable':      var_name,
        'Train N':       len(train),
        'PACF sig lags': str(sig_lags),
        'Lag ceiling':   lag_ceil,
        'Features':      lag_ceil,
    })

print(f'\n{"─"*60}')
print('Step 1 complete — frozen hyperparameters stored for all twelve variables.')

display(pd.DataFrame(lag_summary).style
    .set_caption('Table 4.1 — Lag Selection Summary: All Twelve Variables')
    .hide(axis='index'))


────────────────────────────────────────────────────────────
  us_credit_qoq_growth
────────────────────────────────────────────────────────────
  Significant PACF lags: [1, 2, 6, 8]  →  ceiling: 8
  LASSO        alpha=0.0028
  Ridge        alpha=2.2275
  Elastic Net  alpha=0.0048  l1_ratio=0.5000
  KRR          {'alpha': 0.001, 'gamma': 0.001, 'kernel': 'rbf'}
  SVR          {'C': 1, 'epsilon': 0.1}
  Random Forest {'max_depth': 3, 'min_samples_leaf': 2, 'n_estimators': 100}
  XGBoost      {'learning_rate': 0.1, 'max_depth': 5, 'n_estimators': 100, 'subsample': 0.8}

────────────────────────────────────────────────────────────
  us_sp500_log_ret
────────────────────────────────────────────────────────────
  Significant PACF lags: [7]  →  ceiling: 7
  LASSO        alpha=0.0076
  Ridge        alpha=904.7357
  Elastic Net  alpha=0.0151  l1_ratio=0.5000
  KRR          {'alpha': 10, 'gamma': 0.001, 'kernel': 'polynomial'}
  SVR          {'C': 0.1, 'epsilon': 0.1}
  Random Forest {'max_dep

Variable,Train N,PACF sig lags,Lag ceiling,Features
us_credit_qoq_growth,120,"[1, 2, 6, 8]",8,8
us_sp500_log_ret,279,[7],7,7
us_vix_log_ret,119,"[1, 2]",2,2
us_bond_yield_10y,267,"[1, 4, 6]",6,6
us_oil_yoy,132,"[1, 4, 8]",8,8
us_reer_diff,103,[],2,2
us_gdp_yoy_growth,120,"[1, 2, 5]",5,5
us_cpi,288,"[1, 2, 4, 5, 6]",6,6
us_unemployment,288,"[1, 2, 3, 5, 7]",7,7
us_consumer_confidence,260,"[1, 2, 3, 4, 5, 6, 7, 8]",8,8


### Step 1 Results — Lag Selection Findings
___

Lag ceilings range from 2 (`us_reer_diff`, no PACF lags reached significance — falls back
to the default minimum) to 8 (`us_credit_qoq_growth`, `us_oil_yoy`, `us_consumer_confidence`,
`us_indprod_yoy`), reflecting genuine differences in persistence structure across variable
types. The PACF search window is capped at a fixed 8 lags (2 years of quarterly history)
rather than left open-ended; an earlier version of this analysis used an uncapped search
and found PACF significance climbing all the way to the edge of the search window for
several series (`us_consumer_confidence` showed nearly every lag from 1–20 as significant),
which is the standard symptom of the significance threshold flagging small but statistically
detectable correlations at large N rather than genuine long-memory structure. The fixed
8-lag cap avoids this.

**`us_cpi`** shows a richer autocorrelation structure than most variables (significant lags
at 1, 2, 4, 5, 6), consistent with the well-documented persistence of US inflation discussed
in the stationarity note above. Its LASSO/Elastic Net regularisation is correspondingly low
(alpha=0.0027 / 0.0054) — the models found genuine lagged signal to retain rather than
shrinking aggressively.

**Hyperparameter patterns across the pool**: regularisation strength (LASSO/Elastic Net
alpha) is lowest for series with rich, well-defined lag structure (`us_gdp_yoy_growth`:
alpha=0.0014; `us_unemployment`: alpha=0.0016; `us_cpi`: alpha=0.0027) and highest for
series with weak or sparse autocorrelation (`us_reer_diff`: alpha=0.46; `us_sp500_log_ret`:
alpha=0.011). Ridge and KRR show the same pattern: `us_reer_diff` and `us_sp500_log_ret`
push Ridge alpha to the upper end of the search range (162–905) and KRR alpha/gamma toward
their regularised extremes, indicating these two series have little genuine own-lag signal
for any linear or kernel model to exploit — confirmed independently in Section 5, where
neither beats naive persistence.

**Kernel choice**: KRR selected the polynomial kernel for seven of the twelve variables and
RBF for the remaining five (`us_credit_qoq_growth`, `us_reer_diff`, `us_cpi`,
`us_house_price_yoy`, `us_indprod_yoy`). No single kernel dominates across variable types,
supporting the decision to grid search both rather than fixing one in advance.

## Section 5 — Step 2: Model Evaluation on COVID-Free Test Set
___

**Design**: each model is trained once on the Step 1 frozen hyperparameters and
recursive forecasts are generated for all 16 test quarters from that single fixed model.
No retraining occurs as test-period actuals accumulate. This single-origin static design
directly proxies Step 3 deployment conditions, where 20 quarters are forecast without
any updating from observed actuals.

**YoY base effect caveat**: for the four YoY variables (`us_oil_yoy`, `us_gdp_yoy_growth`,
`us_house_price_yoy`, `us_indprod_yoy`), test target values for 2022 Q1 to 2022 Q4 carry
COVID base effects through the YoY denominator. RMSE and MAE are therefore reported for
both the full 16-quarter window and the 12-quarter sub-window (2023 Q1 to 2025 Q4) that is
free of base effects. The 12-quarter sub-window is the primary basis for model selection
for these variables. For log return, QoQ, level, and first-difference variables, the full
16-quarter window is used as primary.

**Model selection**: among stable models (see the recursive-forecast stability check
below), the winner is the model with the lowest primary-window RMSE *among those that
significantly beat a naive persistence baseline* (Diebold-Mariano stat > 0 and p < 0.05).
For five of twelve variables (`us_credit_qoq_growth`, `us_sp500_log_ret`, `us_vix_log_ret`,
`us_reer_diff`, `us_consumer_confidence`), no model clears this bar — the closest model is
still selected so Step 3 has a model to retrain and forecast with, but it is explicitly
flagged as unvalidated in Table 5.3 and carried through every later table (`Validated`
column in Tables 6.1, 7.1, and 8.1/8.2). This matters because the original lowest-RMSE-only
selection silently picked a "winner" for these five even when nothing actually outperformed
just predicting no change — three of the four borderline cases sit close to the 5%
threshold (`us_sp500_log_ret` p=0.165, `us_vix_log_ret` p=0.077, `us_reer_diff` p=0.073),
consistent with weak-form market efficiency for the two return series and with the lack of
own-lag structure identified for `us_reer_diff` in Section 4.

**Note on baseline**: the Diebold-Mariano test uses a naive random-walk baseline
(last observed value carried forward), not the SARIMA baseline. The Stage 1 SARIMA-vs-ML
model competition occurs in a later assembly step once Danny's SARIMA output is available;
the "winner" and "validated" labels here describe only the ML pool's performance against
naive persistence.

In [13]:
# ─────────────────────────────────────────────────────────────────────────────
# Section 5 — Step 2: Model Evaluation on Test Set (single-origin static)
# ─────────────────────────────────────────────────────────────────────────────

def diebold_mariano(e1, e2):
    """Diebold-Mariano test: e1 vs e2 loss differential. H0: equal accuracy.
    Positive DM stat means e2 (the ML model) has lower squared error than
    e1 (the naive/persistence benchmark) — i.e. the model beats naive."""
    d  = e1**2 - e2**2
    n  = len(d)
    dm = np.mean(d) / (np.std(d, ddof=1) / np.sqrt(n))
    pv = 2 * (1 - stats.norm.cdf(abs(dm)))
    return round(dm, 4), round(pv, 4)

def recursive_forecast(model, seed_series, scaler, n_lags, n_steps):
    """Generate recursive multi-step forecast from a fitted model."""
    history = list(seed_series.values[-n_lags:])
    preds   = []
    for _ in range(n_steps):
        x    = np.array(history[-n_lags:][::-1]).reshape(1, -1)
        x_sc = scaler.transform(x)
        pred = model.predict(x_sc)[0]
        preds.append(pred)
        history.append(pred)
    return np.array(preds)

def is_forecast_stable(preds, train_series, max_multiple=10):
    """Flag forecasts that diverge wildly beyond the training data's range.
    A model is unstable if its recursive forecast drifts more than
    max_multiple times the training data's range away from the training mean.
    This catches RBF/kernel divergence under recursive multi-step forecasting."""
    train_range  = train_series.max() - train_series.min()
    train_center = train_series.mean()
    if train_range == 0:
        train_range = 1.0  # avoid div-by-zero for degenerate series
    pred_max_dev = np.max(np.abs(preds - train_center))
    return pred_max_dev <= max_multiple * train_range

eval_results    = {}
winner_dict     = {}
validated_dict  = {}
eval_rows_all   = []
unstable_log    = []
no_skill_log    = []

for var_name, cfg in VARIABLES.items():
    train  = train_dict[var_name]
    test   = test_dict[var_name]
    models = frozen_models[var_name]
    scaler = scaler_dict[var_name]
    n_lags = lag_dict[var_name]
    is_yoy = cfg['yoy_caveat']

    y_true_full = test.values
    sub_mask    = test.index >= YOY_SUBWINDOW_START
    y_true_sub  = test[sub_mask].values

    sarima_pred = np.full(len(test), train.iloc[-1])

    model_metrics = {}

    for model_name, model in models.items():
        preds_full = recursive_forecast(model, train, scaler, n_lags, len(test))

        stable = is_forecast_stable(preds_full, train)
        if not stable:
            unstable_log.append({
                'Variable': var_name, 'Model': model_name,
                'Max |pred|': round(np.max(np.abs(preds_full)), 2),
                'Train range': round(train.max() - train.min(), 2),
            })

        rmse_full = np.sqrt(mean_squared_error(y_true_full, preds_full))
        mae_full  = mean_absolute_error(y_true_full, preds_full)

        preds_sub = preds_full[sub_mask]
        rmse_sub  = np.sqrt(mean_squared_error(y_true_sub, preds_sub))
        mae_sub   = mean_absolute_error(y_true_sub, preds_sub)

        rmse_primary = rmse_sub if is_yoy else rmse_full

        e_model  = y_true_full - preds_full
        e_sarima = y_true_full - sarima_pred
        dm_stat, dm_pval = diebold_mariano(e_sarima, e_model)

        model_metrics[model_name] = {
            'RMSE (full)':    round(rmse_full, 4),
            'MAE (full)':     round(mae_full, 4),
            'RMSE (primary)': round(rmse_primary, 4),
            'DM stat':        dm_stat,
            'DM pval':        dm_pval,
            'Stable':         stable,
        }

        eval_rows_all.append({
            'Variable':       var_name,
            'Model':          model_name,
            'RMSE (full)':    round(rmse_full, 4),
            'MAE (full)':     round(mae_full, 4),
            'RMSE (primary)': round(rmse_primary, 4),
            'DM stat':        dm_stat,
            'DM pval':        dm_pval,
            'Primary window': '2023 Q1-2025 Q4' if is_yoy else '2022 Q1-2025 Q4',
            'Stable':         'Yes' if stable else 'No — EXCLUDED',
        })

    eval_results[var_name] = model_metrics

    # ── Winner selection ────────────────────────────────────────────────────
    # Among stable models, prefer ones that significantly beat naive
    # persistence (DM stat > 0, DM p-value < 0.05). If none qualify, still
    # select a winner (closest to beating naive) so Section 6 always has a
    # real model to retrain and forecast with — but mark it unvalidated so
    # that flag survives into every later table.
    stable_models = {m: v for m, v in model_metrics.items() if v['Stable']}
    if not stable_models:
        stable_models = model_metrics
        print(f'  WARNING: all models unstable for {var_name} — selecting by RMSE anyway')

    significant_models = {
        m: v for m, v in stable_models.items()
        if v['DM stat'] > 0 and v['DM pval'] < 0.05
    }

    if significant_models:
        winner    = min(significant_models, key=lambda m: significant_models[m]['RMSE (primary)'])
        validated = True
    else:
        winner    = max(stable_models, key=lambda m: stable_models[m]['DM stat'])
        validated = False
        no_skill_log.append({
            'Variable':      var_name,
            'Closest model': winner,
            'DM stat':       stable_models[winner]['DM stat'],
            'DM pval':       stable_models[winner]['DM pval'],
        })

    winner_dict[var_name]    = winner
    validated_dict[var_name] = validated

eval_df = pd.DataFrame(eval_rows_all)

def highlight_winner(row):
    var = row['Variable']
    win = winner_dict.get(var, '')
    if row['Stable'] == 'No — EXCLUDED':
        return ['background-color:#FADBD8; color:#1a1a1a'] * len(row)
    if row['Model'] == win:
        if validated_dict.get(var, False):
            return ['background-color:#D5F5E3; font-weight:bold'] * len(row)
        else:
            return ['background-color:#FEF9E7; font-weight:bold'] * len(row)
    return [''] * len(row)

display(eval_df.style
    .apply(highlight_winner, axis=1)
    .format({'RMSE (full)': '{:.4f}', 'MAE (full)': '{:.4f}',
             'RMSE (primary)': '{:.4f}', 'DM stat': '{:.4f}', 'DM pval': '{:.4f}'})
    .set_caption('Table 5.1 — Step 2 Model Evaluation: All Twelve Variables '
                 '(green = validated winner, yellow = unvalidated winner — '
                 'closest model but does not significantly beat persistence, '
                 'red = excluded for forecast instability)')
    .hide(axis='index'))

print('\nWinner per variable (lowest primary RMSE; * = does not significantly beat naive persistence):')
for var, win in winner_dict.items():
    rmse = eval_results[var][win]['RMSE (primary)']
    flag = '' if validated_dict[var] else '   *unvalidated*'
    print(f'  {var:30s}  →  {win:15s}  (RMSE={rmse:.4f}){flag}')

if no_skill_log:
    print(f'\n{"─"*60}')
    print('Variables where the selected winner does not significantly beat naive persistence:')
    display(pd.DataFrame(no_skill_log).style
        .set_caption('Table 5.3 — Unvalidated Winners (closest model, but DM test not significant)')
        .hide(axis='index'))

if unstable_log:
    print(f'\n{"─"*60}')
    print('Models excluded for forecast instability (recursive divergence):')
    display(pd.DataFrame(unstable_log).style
        .set_caption('Table 5.2 — Excluded Unstable Models')
        .hide(axis='index'))

Variable,Model,RMSE (full),MAE (full),RMSE (primary),DM stat,DM pval,Primary window,Stable
us_credit_qoq_growth,LASSO,0.6446,0.5561,0.6446,-2.4060,0.0161,2022 Q1-2025 Q4,Yes
us_credit_qoq_growth,Ridge,0.6450,0.5569,0.6450,-2.4095,0.0160,2022 Q1-2025 Q4,Yes
us_credit_qoq_growth,Elastic Net,0.6447,0.5562,0.6447,-2.4065,0.0161,2022 Q1-2025 Q4,Yes
us_credit_qoq_growth,KRR,0.7546,0.6724,0.7546,-2.9799,0.0029,2022 Q1-2025 Q4,Yes
us_credit_qoq_growth,SVR,0.9315,0.8488,0.9315,-3.7623,0.0002,2022 Q1-2025 Q4,Yes
us_credit_qoq_growth,Random Forest,0.7124,0.6209,0.7124,-2.5346,0.0113,2022 Q1-2025 Q4,Yes
us_credit_qoq_growth,XGBoost,0.6246,0.4935,0.6246,-1.2673,0.2050,2022 Q1-2025 Q4,Yes
us_sp500_log_ret,LASSO,0.0755,0.0623,0.0755,1.3833,0.1666,2022 Q1-2025 Q4,Yes
us_sp500_log_ret,Ridge,0.0756,0.0623,0.0756,1.3886,0.1649,2022 Q1-2025 Q4,Yes
us_sp500_log_ret,Elastic Net,0.0755,0.0623,0.0755,1.3835,0.1665,2022 Q1-2025 Q4,Yes



Winner per variable (lowest primary RMSE; * = does not significantly beat naive persistence):
  us_credit_qoq_growth            →  XGBoost          (RMSE=0.6246)   *unvalidated*
  us_sp500_log_ret                →  Ridge            (RMSE=0.0756)   *unvalidated*
  us_vix_log_ret                  →  SVR              (RMSE=0.2189)   *unvalidated*
  us_bond_yield_10y               →  LASSO            (RMSE=1.5064)
  us_oil_yoy                      →  Elastic Net      (RMSE=23.5219)
  us_reer_diff                    →  KRR              (RMSE=2.7807)   *unvalidated*
  us_gdp_yoy_growth               →  XGBoost          (RMSE=0.4052)
  us_cpi                          →  Ridge            (RMSE=2.7227)
  us_unemployment                 →  Random Forest    (RMSE=0.3382)
  us_consumer_confidence          →  XGBoost          (RMSE=0.6212)   *unvalidated*
  us_house_price_yoy              →  Ridge            (RMSE=1.8696)
  us_indprod_yoy                  →  Random Forest    (RMSE=1.8802)

───────

Variable,Closest model,DM stat,DM pval
us_credit_qoq_growth,XGBoost,-1.267300,0.205000
us_sp500_log_ret,Ridge,1.388600,0.164900
us_vix_log_ret,SVR,1.766200,0.077400
us_reer_diff,KRR,1.791900,0.073100
us_consumer_confidence,XGBoost,-0.442600,0.658100


## Section 6 — Step 3: Final 20-Quarter Forecast
___

**Design**: for each variable, the winning model class selected in Step 2 (Table 5.1) is
re-tuned from scratch on the full COVID-excluded dataset — training and test periods
combined — rather than reusing the Step 1 frozen hyperparameters. This gives the final
forecasting model access to the most recent four years of data (2022 Q1–2025 Q4) during
hyperparameter selection, which Step 1's training cutoff (2019 Q4) deliberately withheld
to keep Step 2 a genuine out-of-sample test.

**Re-tuning procedure**: regularised linear models (LASSO, Ridge, Elastic Net) re-run
their built-in cross-validated alpha search; KRR, SVR, Random Forest, and XGBoost repeat
the same `GridSearchCV` search space used in Step 1 (Section 4), now fit against the
larger combined dataset via `TimeSeriesSplit`. The model class itself (e.g. "Random
Forest") is fixed from Step 2; only its hyperparameters are re-selected.

**Forecast generation**: each re-tuned model produces a 20-quarter recursive forecast
(2026 Q1 to 2030 Q4) using `recursive_forecast` — the same function used for Step 2
evaluation, feeding each prediction back in as a lagged input for the next quarter. No
exogenous scenario inputs are used; this is a pure univariate continuation of each
variable's own historical pattern.

**Validation status carried forward**: the `Validated` flag from Step 2 (Table 5.3) is
attached to every forecast in Table 6.1. Five variables (`us_credit_qoq_growth`,
`us_sp500_log_ret`, `us_vix_log_ret`, `us_reer_diff`, `us_consumer_confidence`) are
forecast using the closest-performing model from Step 2 despite that model not
significantly beating naive persistence — these forecasts are retained so all twelve
variables produce a complete output for the Stage 1 model competition, but they should be
treated as low-confidence relative to the seven validated variables. This flag persists
through Section 7's back-transformation and Section 8's final assembled output.

**Limitation**: statistical validation in Step 2 reflects performance on a fixed
16-quarter historical test window, not on a 20-quarter unconditional extrapolation.
A model can be validated against persistence in Step 2 and still show implausible
recursive drift over the longer Step 3 horizon (see the `us_oil_yoy` and `us_unemployment`
discussion below) — these are two separate claims, and only the first is directly tested
here.

In [15]:
# ─────────────────────────────────────────────────────────────────────────────
# Section 6 — Step 3: Final 20-Quarter Forecast
# ─────────────────────────────────────────────────────────────────────────────
# Retune hyperparameters on full COVID-excluded dataset (training + test).
# Fit final model per variable. Recursive forecast 2026 Q1 to 2030 Q4.
# ─────────────────────────────────────────────────────────────────────────────

from sklearn.model_selection import cross_val_score

def build_model_from_winner(winner_name, var_name, frozen):
    """Return a fresh unfitted model instance with Step 1 frozen hyperparameters."""
    m = frozen[var_name][winner_name]
    if winner_name == 'LASSO':
        from sklearn.linear_model import Lasso
        return Lasso(alpha=m.alpha_, max_iter=10000)
    elif winner_name == 'Ridge':
        from sklearn.linear_model import Ridge
        return Ridge(alpha=m.alpha_)
    elif winner_name == 'Elastic Net':
        from sklearn.linear_model import ElasticNet
        return ElasticNet(alpha=m.alpha_, l1_ratio=m.l1_ratio_, max_iter=10000)
    elif winner_name == 'KRR':
        return KernelRidge(
            alpha=m.alpha, kernel=m.kernel,
            gamma=m.gamma if hasattr(m, 'gamma') else None)
    elif winner_name == 'SVR':
        return SVR(kernel='rbf', C=m.C, epsilon=m.epsilon)
    elif winner_name == 'Random Forest':
        return RandomForestRegressor(
            n_estimators=m.n_estimators, max_depth=m.max_depth,
            min_samples_leaf=m.min_samples_leaf, random_state=RANDOM_STATE)
    elif winner_name == 'XGBoost':
        return XGBRegressor(
            max_depth=m.max_depth, learning_rate=m.learning_rate,
            n_estimators=m.n_estimators, subsample=m.subsample,
            random_state=RANDOM_STATE, verbosity=0)

# Build forecast index — 2026 Q1 to 2030 Q4
forecast_index = pd.date_range(
    start='2026-03-31', periods=FORECAST_HORIZON, freq='QE-DEC')

final_models   = {}
final_scalers  = {}
forecast_dict  = {}
step3_summary  = []

for var_name, cfg in VARIABLES.items():
    winner = winner_dict[var_name]
    excl   = excl_dict[var_name]
    n_lags = lag_dict[var_name]

    # Full COVID-excluded dataset (train + test combined)
    full_data = excl[excl.index <= TEST_END].copy()

    # Build feature matrix on full dataset
    full_df   = make_features(full_data, n_lags)
    X_full    = full_df.drop(columns='y').values
    y_full    = full_df['y'].values

    # Re-tune scaler on full dataset
    scaler_full = StandardScaler()
    X_full_sc   = scaler_full.fit_transform(X_full)

    # Re-tune hyperparameters via TimeSeriesSplit on full dataset
    tscv = TimeSeriesSplit(n_splits=CV_SPLITS)

    if winner == 'LASSO':
        final_m = LassoCV(cv=tscv, max_iter=10000, random_state=RANDOM_STATE)
        final_m.fit(X_full_sc, y_full)
    elif winner == 'Ridge':
        final_m = RidgeCV(alphas=np.logspace(-2, 4, 24), cv=tscv)
        final_m.fit(X_full_sc, y_full)
    elif winner == 'Elastic Net':
        final_m = ElasticNetCV(cv=tscv, max_iter=10000, random_state=RANDOM_STATE)
        final_m.fit(X_full_sc, y_full)
    elif winner == 'KRR':
        g = GridSearchCV(KernelRidge(),
            {'alpha': [0.001, 0.01, 0.1, 1, 10],
             'kernel': ['rbf', 'polynomial'],
             'gamma': [0.0001, 0.001, 0.01, 0.1, 1]},
            cv=tscv, scoring='neg_mean_squared_error')
        g.fit(X_full_sc, y_full)
        final_m = g.best_estimator_
    elif winner == 'SVR':
        g = GridSearchCV(SVR(kernel='rbf'),
            {'C': [0.1, 1, 10, 100, 1000, 10000], 'epsilon': [0.01, 0.1, 1]},
            cv=tscv, scoring='neg_mean_squared_error')
        g.fit(X_full_sc, y_full)
        final_m = g.best_estimator_
    elif winner == 'Random Forest':
        g = GridSearchCV(
            RandomForestRegressor(random_state=RANDOM_STATE),
            {'n_estimators': [100, 200], 'max_depth': [3, 5, None],
             'min_samples_leaf': [2, 5]},
            cv=tscv, scoring='neg_mean_squared_error')
        g.fit(X_full_sc, y_full)
        final_m = g.best_estimator_
    elif winner == 'XGBoost':
        g = GridSearchCV(
            XGBRegressor(random_state=RANDOM_STATE, verbosity=0),
            {'max_depth': [3, 5], 'learning_rate': [0.05, 0.1],
             'n_estimators': [100, 200], 'subsample': [0.8, 1.0]},
            cv=tscv, scoring='neg_mean_squared_error')
        g.fit(X_full_sc, y_full)
        final_m = g.best_estimator_

    final_models[var_name]  = final_m
    final_scalers[var_name] = scaler_full

    # Generate 20-quarter recursive forecast
    forecast = recursive_forecast(
        final_m, full_data, scaler_full, n_lags, FORECAST_HORIZON)
    forecast_dict[var_name] = pd.Series(forecast, index=forecast_index)

    validated = validated_dict[var_name]

    step3_summary.append({
        'Variable':      var_name,
        'Winner model':  winner,
        'Validated':     'Yes' if validated else 'No',
        'Full train N':  len(full_data),
        'Forecast mean': round(forecast.mean(), 4),
        'Forecast min':  round(forecast.min(), 4),
        'Forecast max':  round(forecast.max(), 4),
    })

    flag = '' if validated else '   *unvalidated — forecast uses closest model, not a confirmed skill model*'
    print(f'  {var_name:30s}  winner={winner:15s}  '
          f'fcst mean={forecast.mean():.3f}{flag}')

step3_df = pd.DataFrame(step3_summary)

def highlight_unvalidated(row):
    if row['Validated'] == 'No':
        return ['background-color:#FEF9E7; color:#1a1a1a'] * len(row)
    return [''] * len(row)

display(step3_df.style
    .apply(highlight_unvalidated, axis=1)
    .set_caption('Table 6.1 — Step 3 Final Forecast Summary: All Twelve Variables '
                 '(yellow = unvalidated winner from Step 2 — see Table 5.3)')
    .hide(axis='index'))

  us_credit_qoq_growth            winner=XGBoost          fcst mean=0.969   *unvalidated — forecast uses closest model, not a confirmed skill model*
  us_sp500_log_ret                winner=Ridge            fcst mean=0.018   *unvalidated — forecast uses closest model, not a confirmed skill model*
  us_vix_log_ret                  winner=SVR              fcst mean=0.011   *unvalidated — forecast uses closest model, not a confirmed skill model*
  us_bond_yield_10y               winner=LASSO            fcst mean=4.435
  us_oil_yoy                      winner=Elastic Net      fcst mean=9.063
  us_reer_diff                    winner=KRR              fcst mean=0.149   *unvalidated — forecast uses closest model, not a confirmed skill model*
  us_gdp_yoy_growth               winner=XGBoost          fcst mean=2.409
  us_cpi                          winner=Ridge            fcst mean=3.233
  us_unemployment                 winner=Random Forest    fcst mean=5.603
  us_consumer_confidence          

Variable,Winner model,Validated,Full train N,Forecast mean,Forecast min,Forecast max
us_credit_qoq_growth,XGBoost,No,136,0.968700,0.798800,1.278000
us_sp500_log_ret,Ridge,No,295,0.017800,0.013800,0.021300
us_vix_log_ret,SVR,No,135,0.011000,-0.021200,0.046100
us_bond_yield_10y,LASSO,Yes,283,4.435300,4.117400,4.705000
us_oil_yoy,Elastic Net,Yes,148,9.062900,-4.950400,17.565800
us_reer_diff,KRR,No,119,0.149100,0.149100,0.149300
us_gdp_yoy_growth,XGBoost,Yes,136,2.409200,1.880100,3.103400
us_cpi,Ridge,Yes,304,3.233000,2.818900,3.475000
us_unemployment,Random Forest,Yes,304,5.603000,4.724900,6.520800
us_consumer_confidence,XGBoost,No,276,100.600998,100.217400,100.932098


### Step 3 Results — Final 20-Quarter Forecast Summary (2026 Q1 – 2030 Q4)

Step 3 re-tunes hyperparameters on the full COVID-excluded dataset (training + test
combined) and generates recursive 20-quarter forecasts for each variable using the
winning model selected in Step 2, carrying forward the `Validated` flag from Table 5.3.

**Forecast plausibility assessment (all twelve variables):**

| Variable | Winner | Validated | Forecast mean | Assessment |
|---|---|---|---|---|
| us_credit_qoq_growth | XGBoost | No | 0.97% | Plausible level, but winner did not significantly beat persistence in Step 2 |
| us_sp500_log_ret | Ridge | No | 0.018 | Plausible modest positive return, but not a significant improvement over naive |
| us_vix_log_ret | SVR | No | 0.011 | Plausible, but not a significant improvement over naive |
| us_bond_yield_10y | LASSO | Yes | 4.44% | Plausible — consistent with current rate environment |
| us_oil_yoy | Elastic Net | Yes | 9.06% | Validated in Step 2, but recursive YoY compounding implies oil prices nearly doubling by 2030 — flagged as a back-transformation fragility, not necessarily a realistic path |
| us_reer_diff | KRR | No | 0.149 | Flat forecast (0.1491–0.1493 across all 20 quarters) — consistent with the near-zero own-lag signal identified in Section 4 |
| us_gdp_yoy_growth | XGBoost | Yes | 2.41% | Plausible, stable around long-run trend |
| us_cpi | Ridge | Yes | 3.23% | Plausible, consistent with recent inflation readings |
| us_unemployment | Random Forest | Yes | 5.60% | Validated in Step 2, but climbs from a 4.40% anchor with no shock — recursive drift, treat with caution |
| us_consumer_confidence | XGBoost | No | 100.6 | Near-flat forecast; also did not significantly beat persistence in Step 2 |
| us_house_price_yoy | Ridge | Yes | 0.53% | Plausible, modest growth |
| us_indprod_yoy | Random Forest | Yes | 1.63% | Validated in Step 2, but forecast is nearly flat (1.586–1.636) across the horizon — tree-based recursive forecasting saturating rather than extrapolating |

**Note on the five unvalidated variables**: `us_credit_qoq_growth`, `us_sp500_log_ret`,
`us_vix_log_ret`, `us_reer_diff`, and `us_consumer_confidence` are forecast using the
closest-performing model from Step 2, but none of these models significantly beat naive
persistence (see Table 5.3). These forecasts are retained for completeness and to keep
the dataset shape consistent for the Stage 1 model competition, but should be treated as
low-confidence relative to the other seven variables.

**Note on `us_reer_diff`**: the forecast is effectively constant across all 20 quarters.
This is a direct consequence of the near-zero own-lag signal found in Section 4 (every
linear and kernel model pushed toward maximum regularisation for this variable) and is
consistent with it failing the DM significance test in Step 2 — not a new error, but the
same underlying finding appearing for a third time across three independent diagnostics.

**Note on `us_consumer_confidence` and `us_indprod_yoy`**: both show near-flat forecasts
despite using tree-based winners (XGBoost and Random Forest respectively). This is a
separate phenomenon from the `us_reer_diff` case — it reflects a known limitation of
recursive multi-step forecasting with tree ensembles, which cannot extrapolate beyond the
range of values seen in training and tend to saturate to a near-constant prediction once
the recursive feature values drift outside that range. `us_indprod_yoy` is statistically
validated against persistence in Step 2; `us_consumer_confidence` is not — these are
independent findings that happen to coincide for `us_consumer_confidence` but not for
`us_indprod_yoy`.

**Note on `us_oil_yoy` and `us_unemployment`**: both are validated against persistence in
Step 2, but their 20-quarter forecast paths show signs of recursive extrapolation drift
that the DM test does not check for — `us_oil_yoy`'s YoY forecast compounds recursively on
itself and implies the oil price level nearly doubling by 2030; `us_unemployment` climbs
from a 4.40% anchor to a forecast mean of 5.60% with no corresponding economic shock in
the model. Statistical validation against historical persistence is a different claim
from extrapolation reliability over a 20-quarter horizon, and both should be flagged as
limitations in Section 7/8 of the report.

**Note on `us_bond_yield_10y`**: this variable is forecast in levels despite failing
stationarity tests (ADF p=0.46, KPSS p=0.01). This is a deliberate methodological
exception — the bond yield is trend-stationary rather than difference-stationary, and
IFRS 9 requires the yield to be reported in levels for use as a PD model regressor. This
is consistent with the treatment in `01_EDA.ipynb` and `02a_Stage1_TimeSeries.ipynb`.

In [17]:
# ─────────────────────────────────────────────────────────────────────────────
# Section 7 — Back-transformation to Implied Levels
# ─────────────────────────────────────────────────────────────────────────────
# Reconstruct implied level paths from stationary forecasts using the last
# observed actual value as anchor. Required for Stage B PD model regressors.
# ─────────────────────────────────────────────────────────────────────────────

level_forecast_dict = {}
backtransform_summary = []

for var_name, cfg in VARIABLES.items():
    transform   = cfg['transform']
    forecast    = forecast_dict[var_name]
    raw_col     = cfg['raw_col']

    # Last observed raw level (anchor point — 2025 Q4)
    raw_series  = raw[raw_col].dropna()
    raw_series  = raw_series[raw_series.index <= DATA_CUTOFF]
    last_level  = raw_series.iloc[-1]
    last_date   = raw_series.index[-1]

    if transform == 'level':
        level_path = forecast.copy()

    elif transform == 'log_ret':
        levels = [last_level]
        for lr in forecast.values:
            levels.append(levels[-1] * np.exp(lr))
        level_path = pd.Series(levels[1:], index=forecast.index)

    elif transform == 'qoq':
        levels = [last_level]
        for qoq in forecast.values:
            levels.append(levels[-1] * (1 + qoq / 100))
        level_path = pd.Series(levels[1:], index=forecast.index)

    elif transform == 'yoy':
        history = list(raw_series.iloc[-4:].values)
        levels  = []
        for i, yoy in enumerate(forecast.values):
            if i < 4:
                denom = history[i]
            else:
                denom = levels[i-4]
            new_level = denom * (1 + yoy / 100)
            levels.append(new_level)
        level_path = pd.Series(levels, index=forecast.index)

    elif transform == 'diff':
        reer_interp = raw_series.interpolate(method='linear', limit_direction='both')
        last_reer   = reer_interp.iloc[-1]
        levels = [last_reer]
        for d in forecast.values:
            levels.append(levels[-1] + d)
        level_path = pd.Series(levels[1:], index=forecast.index)

    level_forecast_dict[var_name] = level_path

    backtransform_summary.append({
        'Variable':        var_name,
        'Transform':       transform,
        'Validated':       'Yes' if validated_dict[var_name] else 'No',
        'Anchor value':    round(last_level, 4),
        'Anchor date':     last_date.date(),
        'Level fcst mean': round(level_path.mean(), 4),
        'Level fcst min':  round(level_path.min(), 4),
        'Level fcst max':  round(level_path.max(), 4),
    })

backtransform_df = pd.DataFrame(backtransform_summary)

def highlight_unvalidated_bt(row):
    if row['Validated'] == 'No':
        return ['background-color:#FEF9E7; color:#1a1a1a'] * len(row)
    return [''] * len(row)

display(backtransform_df.style
    .apply(highlight_unvalidated_bt, axis=1)
    .set_caption('Table 7.1 — Back-transformation to Implied Levels: All Twelve Variables '
                 '(yellow = unvalidated winner from Step 2 — see Table 5.3)')
    .hide(axis='index'))

Variable,Transform,Validated,Anchor value,Anchor date,Level fcst mean,Level fcst min,Level fcst max
us_credit_qoq_growth,qoq,No,43143.826000,2025-12-31,47886.887900,43488.479000,52317.501200
us_sp500_log_ret,log_ret,No,6845.500000,2025-12-31,8267.413800,6953.248300,9774.887800
us_vix_log_ret,log_ret,No,14.950000,2025-12-31,16.914100,15.326200,18.621800
us_bond_yield_10y,level,Yes,4.140000,2025-12-31,4.435300,4.117400,4.705000
us_oil_yoy,yoy,Yes,57.260000,2025-12-31,84.532800,67.318200,101.628000
us_reer_diff,diff,No,106.710000,2025-12-31,108.275900,106.859100,109.692600
us_gdp_yoy_growth,yoy,Yes,24055.749000,2025-12-31,25639.500400,24062.387900,27137.525600
us_cpi,level,Yes,2.653300,2025-12-31,3.233000,2.818900,3.475000
us_unemployment,level,Yes,4.400000,2025-12-31,5.603000,4.724900,6.520800
us_consumer_confidence,level,No,100.264800,2025-12-31,100.600998,100.217400,100.932100


In [21]:
# ─────────────────────────────────────────────────────────────────────────────
# Section 8 — Assembly & Output
# ─────────────────────────────────────────────────────────────────────────────

# ── Build output dataframe — stationary transformed forecasts ─────────────────
forecast_df = pd.DataFrame(forecast_dict)
forecast_df.index = forecast_df.index.date
forecast_df.index.name = 'date'

print(f'Forecast dataframe shape: {forecast_df.shape}')
print(f'Index: {forecast_df.index.min()} to {forecast_df.index.max()}')
print(f'Columns: {forecast_df.columns.tolist()}')

unvalidated_cols = [v for v in forecast_df.columns if not validated_dict[v]]

def highlight_unvalidated_columns(df):
    styles = pd.DataFrame('', index=df.index, columns=df.columns)
    for col in unvalidated_cols:
        styles[col] = 'background-color:#FEF9E7; color:#1a1a1a'
    return styles

display(forecast_df.round(4).style
    .apply(highlight_unvalidated_columns, axis=None)
    .set_caption('Table 8.1 — ML_regressors_US_Q: Final 20-Quarter Stationary Forecasts '
                 '(yellow columns = winner did not significantly beat naive persistence '
                 'in Step 2 — see Table 5.3)')
    .format('{:.4f}'))

# ── Metadata table — winner model and validation status per variable ─────────
metadata_df = pd.DataFrame([
    {
        'Variable':    var_name,
        'Winner model': winner_dict[var_name],
        'Validated':    'Yes' if validated_dict[var_name] else 'No',
        'Lag ceiling':  lag_dict[var_name],
    }
    for var_name in VARIABLES
])

display(metadata_df.style
    .apply(lambda row: ['background-color:#FEF9E7; color:#1a1a1a'] * len(row)
           if row['Validated'] == 'No' else [''] * len(row), axis=1)
    .set_caption('Table 8.2 — Model Metadata: Winner & Validation Status per Variable')
    .hide(axis='index'))

if unvalidated_cols:
    print(f'\n{"─"*60}')
    print(f'WARNING: {len(unvalidated_cols)} of {len(forecast_df.columns)} variables are '
          f'forecast using an unvalidated model (did not significantly beat naive '
          f'persistence in Step 2): {", ".join(unvalidated_cols)}')
    print('These forecasts are included for completeness but should be flagged as '
          'low-confidence in any downstream use (e.g. Stage 2 PD modelling, SARIMA-vs-ML '
          'competition) — see Table 5.3 for DM test details.')

# ── Save outputs ───────────────────────────────────────────────────────────────
forecast_df.to_csv('ML_regressors_US_Q.csv')
metadata_df.to_csv('ML_regressors_US_Q_metadata.csv', index=False)

print(f'\nSaved: ML_regressors_US_Q.csv  (shape={forecast_df.shape})')
print(f'Saved: ML_regressors_US_Q_metadata.csv  (shape={metadata_df.shape})')

Forecast dataframe shape: (20, 12)
Index: 2026-03-31 to 2030-12-31
Columns: ['us_credit_qoq_growth', 'us_sp500_log_ret', 'us_vix_log_ret', 'us_bond_yield_10y', 'us_oil_yoy', 'us_reer_diff', 'us_gdp_yoy_growth', 'us_cpi', 'us_unemployment', 'us_consumer_confidence', 'us_house_price_yoy', 'us_indprod_yoy']


,us_credit_qoq_growth,us_sp500_log_ret,us_vix_log_ret,us_bond_yield_10y,us_oil_yoy,us_reer_diff,us_gdp_yoy_growth,us_cpi,us_unemployment,us_consumer_confidence,us_house_price_yoy,us_indprod_yoy
date,,,,,,,,,,,,
2026-03-31,0.7988,0.0156,0.0461,4.1174,-4.9504,0.1491,2.1835,2.8891,4.7249,100.4751,-1.2941,1.5864
2026-06-30,0.8668,0.0150,-0.0212,4.1488,3.6381,0.1493,2.1226,2.9112,4.9524,100.6654,-1.1122,1.6360
2026-09-30,1.0580,0.0183,0.0241,4.1760,10.9919,0.1491,2.1294,2.8189,5.4729,100.6901,-0.7939,1.6360
2026-12-31,1.1198,0.0213,0.0012,4.2413,17.5658,0.1491,1.9477,3.0380,5.9912,100.8465,-0.5953,1.6360
2027-03-31,0.9107,0.0138,0.0149,4.2745,16.1745,0.1491,2.3315,3.0328,6.2602,100.8700,-0.3948,1.6360
2027-06-30,1.0976,0.0152,0.0078,4.3118,16.0366,0.1491,2.4939,3.0769,6.4945,100.8261,-0.2713,1.6360
2027-09-30,1.1311,0.0181,0.0118,4.3462,11.3768,0.1491,2.6411,3.1982,6.5208,100.8406,-0.1420,1.6360
2027-12-31,0.8291,0.0186,0.0097,4.3779,11.2282,0.1491,2.7274,3.1831,6.3645,100.9321,0.1102,1.6360
2028-03-31,1.2780,0.0186,0.0108,4.4047,11.1499,0.1491,2.8584,3.2409,6.1919,100.8552,0.3782,1.6360


Variable,Winner model,Validated,Lag ceiling
us_credit_qoq_growth,XGBoost,No,8
us_sp500_log_ret,Ridge,No,7
us_vix_log_ret,SVR,No,2
us_bond_yield_10y,LASSO,Yes,6
us_oil_yoy,Elastic Net,Yes,8
us_reer_diff,KRR,No,2
us_gdp_yoy_growth,XGBoost,Yes,5
us_cpi,Ridge,Yes,6
us_unemployment,Random Forest,Yes,7
us_consumer_confidence,XGBoost,No,8



────────────────────────────────────────────────────────────
These forecasts are included for completeness but should be flagged as low-confidence in any downstream use (e.g. Stage 2 PD modelling, SARIMA-vs-ML competition) — see Table 5.3 for DM test details.

Saved: ML_regressors_US_Q.csv  (shape=(20, 12))
Saved: ML_regressors_US_Q_metadata.csv  (shape=(12, 4))
